# SSHC Demo: Cross-Modal Hashing on MIR-Flickr-25k

Colab demo for the seven trained 128-bit hashing models released at
`HF_REPO_ID` below. Cells below regenerate the I→T / T→I symmetry
scatter plot from each model's `eval.json` and run single-image inference
(hash code + retrieval-based multi-label classification) for any image
you upload.

Models

1. `alexnet_dcmh` — DCMH on AlexNet+BoW
2. `alexnet_cmshc` — CM-SHC on AlexNet+BoW
3. `clip_frozen_dcmh` — DCMH on CLIP frozen
4. `clip_frozen_cmshc` — CM-SHC on CLIP frozen
5. `clip_lora_dcmh` — DCMH on CLIP+LoRA
6. `clip_lora_cmshc` — CM-SHC on CLIP+LoRA
7. `clip_lora_anchored_lc30` — Anchored-DCMH (λ_c=3) on CLIP+LoRA

## 1. Setup
Edit the two constants below before running.

In [ ]:
# === EDIT THESE ===
HF_REPO_ID = "YOUR_USER/sshc-mirflickr25k-128bit"
GITHUB_REPO = "https://github.com/YOUR_USER/SSHC.git"
# ==================

In [ ]:
import sys, subprocess

def pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch", "torchvision", "transformers>=4.30", "peft>=0.8",
    "huggingface_hub>=0.20", "omegaconf>=2.3", "pillow", "matplotlib")

# Install the SSHC package (provides DCMH/CMSHC model classes + backbone registry)
pip(f"git+{GITHUB_REPO}")

In [ ]:
import json, io
from pathlib import Path

import numpy as np
import torch
import yaml
from PIL import Image
from huggingface_hub import snapshot_download
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

REPO_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, repo_type="model"))
print("downloaded:", REPO_DIR)
print("contents:", sorted(p.name for p in REPO_DIR.iterdir()))

In [ ]:
# Discover the seven model folders dynamically.
MODEL_KEYS = sorted(
    p.name for p in REPO_DIR.iterdir()
    if p.is_dir() and p.name != "shared" and (p / "model.pt").exists()
)
MODEL_KEYS

## 2. Symmetry scatter (T→I vs I→T MAP)
Reads `eval.json` from each model folder. No model loading needed.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Group models into the three backbone panels matching the report.
PANELS = [
    ("AlexNet + BoW",      ["alexnet_dcmh", "alexnet_cmshc"]),
    ("CLIP ViT-B/32 frozen", ["clip_frozen_dcmh", "clip_frozen_cmshc"]),
    ("CLIP + LoRA",          ["clip_lora_dcmh", "clip_lora_cmshc", "clip_lora_anchored_lc30"]),
]

STYLE = {
    "alexnet_dcmh":            ("#c41e3a", "o", "DCMH"),
    "alexnet_cmshc":           ("#0f9480", "s", "CM-SHC"),
    "clip_frozen_dcmh":        ("#c41e3a", "o", "DCMH"),
    "clip_frozen_cmshc":       ("#0f9480", "s", "CM-SHC"),
    "clip_lora_dcmh":          ("#c41e3a", "o", "DCMH"),
    "clip_lora_cmshc":         ("#0f9480", "s", "CM-SHC"),
    "clip_lora_anchored_lc30": ("#1a2744", "D", "Anchored-DCMH (lc=3)"),
}

def load_xy(key):
    d = json.loads((REPO_DIR / key / "eval.json").read_text())
    m = d.get("map", d)
    return float(m["text_to_image"]), float(m["image_to_text"])

all_xy = [load_xy(k) for k in MODEL_KEYS]
lo = min(min(x, y) for x, y in all_xy) - 0.02
hi = max(max(x, y) for x, y in all_xy) + 0.02

fig, axes = plt.subplots(1, len(PANELS), figsize=(15, 4.5), sharex=True, sharey=True)
for ax, (title, keys) in zip(axes, PANELS):
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="#b9c5d0", linewidth=1)
    for k in keys:
        if k not in MODEL_KEYS:
            continue
        x, y = load_xy(k)
        c, m, label = STYLE[k]
        ax.scatter([x], [y], s=140, color=c, marker=m, edgecolors="white",
                   linewidths=1.0, zorder=3, label=label)
        ax.annotate(label, xy=(x, y), xytext=(x + 0.004, y + 0.006),
                    fontsize=8.5, color="#1a2744")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("T → I mean MAP")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("I → T mean MAP")
fig.suptitle("Cross-modal retrieval symmetry across backbones (128 bits)", fontsize=12)
fig.tight_layout()
plt.show()

## 3. Build the seven models
Uses SSHC's `build_model` so backbones / LoRA adapters are wired correctly.

In [ ]:
from omegaconf import OmegaConf
from src.pipelines.train import build_model, resolve_text_backbone_spec  # noqa

def load_one(key):
    folder = REPO_DIR / key
    cfg = OmegaConf.load(folder / "config.yaml")
    # The config files reference base.yaml / model/{name}.yaml relative to the
    # SSHC repo root. SSHC's loader handles that transparently when you point
    # it at the original config; here we re-load through the package.
    from src.utils.config import load_experiment
    cfg = load_experiment(folder / "config.yaml")
    text_ref, hf_lfo, _ = resolve_text_backbone_spec(cfg)
    model = build_model(cfg, text_ref=text_ref, hf_local_files_only=hf_lfo).to(device).eval()
    ckpt = torch.load(folder / "model.pt", map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
    card = json.loads((folder / "card.json").read_text())
    return {"model": model, "cfg": cfg, "card": card}

models = {}
for k in MODEL_KEYS:
    print(f"loading {k} …")
    models[k] = load_one(k)
print("ready")

## 4. Single-image inference

Each model takes the same ImageNet-normalized 224×224 image (the CLIP
backbones renormalize internally). For classification we use a small
exemplar database shipped in `shared/exemplars/`:

1. encode each exemplar image with the chosen model → 128-bit signed code
2. encode the query → 128-bit signed code
3. find the top-K nearest exemplars by Hamming distance
4. average their multi-hot labels and threshold at 0.5

The exemplar codes are cached per-model after the first call so subsequent
queries are fast.

In [ ]:
# Load shared exemplar set (200 sample images with labels)
EX_DIR = REPO_DIR / "shared" / "exemplars"
EX_LABELS = np.load(EX_DIR / "labels.npy") if (EX_DIR / "labels.npy").exists() else None
EX_INDEX = (
    json.loads((EX_DIR / "index.json").read_text())
    if (EX_DIR / "index.json").exists() else []
)
LABEL_NAMES = json.loads((REPO_DIR / "shared" / "label_names.json").read_text())
print(f"exemplars: {len(EX_INDEX)}; labels: {len(LABEL_NAMES)}")

In [ ]:
PREPROC = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def to_tensor(img):
    if isinstance(img, (str, Path)):
        img = Image.open(img).convert("RGB")
    return PREPROC(img).unsqueeze(0).to(device)

@torch.no_grad()
def encode_image(model, x):
    out = model.encode_image(x) if hasattr(model, "encode_image") else model(image=x)
    return out

_exemplar_cache = {}

@torch.no_grad()
def get_exemplar_codes(key):
    if key in _exemplar_cache:
        return _exemplar_cache[key]
    if EX_LABELS is None:
        raise RuntimeError("no exemplars uploaded; classification disabled")
    model = models[key]["model"]
    codes = []
    for entry in EX_INDEX:
        x = to_tensor(EX_DIR / "images" / entry["filename"])
        h = encode_image(model, x)
        codes.append(torch.sign(h).cpu())
    codes = torch.cat(codes, dim=0)  # (N, 128)
    _exemplar_cache[key] = codes
    return codes

def classify(key, image, top_k=10, threshold=0.5):
    """Return (hash_code, predicted_label_names, top_k_neighbor_indices)."""
    model = models[key]["model"]
    x = to_tensor(image)
    with torch.no_grad():
        h = encode_image(model, x)
        h = torch.sign(h).cpu().squeeze(0)  # (128,)
    if EX_LABELS is None:
        return h.numpy(), [], np.array([])
    db_codes = get_exemplar_codes(key)
    hamming = (h.unsqueeze(0) != db_codes).sum(dim=1).numpy()  # (N,)
    top_idx = np.argsort(hamming)[:top_k]
    aggregate = EX_LABELS[top_idx].mean(axis=0)
    pred_mask = aggregate >= threshold
    pred_names = [LABEL_NAMES[i] for i, m in enumerate(pred_mask) if m]
    return h.numpy(), pred_names, top_idx

### Run inference on an image
Either upload one in Colab (the snippet below) or pass a local path.

In [ ]:
# Option A: upload from your computer
try:
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
except Exception:
    # Option B: fall back to a sample exemplar bundled in the repo
    image_path = str(EX_DIR / "images" / EX_INDEX[0]["filename"])
    print("using bundled exemplar:", image_path)

In [ ]:
from PIL import Image as _PILImage

img = _PILImage.open(image_path).convert("RGB")
plt.figure(figsize=(3.5, 3.5)); plt.imshow(img); plt.axis("off"); plt.show()

for key in MODEL_KEYS:
    h, labels, top_idx = classify(key, img, top_k=10)
    bits = "".join("1" if v > 0 else "0" for v in h)
    short = bits[:32] + "…" + bits[-16:]
    label_str = ", ".join(labels) if labels else "(none above threshold)"
    print(f"\n--- {key} ({models[key]['card']['loss']}) ---")
    print(f"  hash[0:32]…[-16:]: {short}")
    print(f"  predicted labels:  {label_str}")

## Optional: cross-model agreement
How often do the seven models agree on a label?

In [ ]:
from collections import Counter

votes = Counter()
for key in MODEL_KEYS:
    _, labels, _ = classify(key, img, top_k=10)
    votes.update(labels)

if votes:
    print("label : models that predicted it (out of 7)")
    for lbl, n in votes.most_common():
        print(f"  {lbl:14s} {n}/7")
else:
    print("no labels above threshold from any model")